This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision diffusers transformers accelerate --upgrade -q

## Image generation

### Deep learning for image generation

#### Sampling from latent spaces of images

#### Variational autoencoders

#### Implementing a VAE with Keras

In [ ]:
# PyTorch: the functional encoder becomes an nn.Module. Keras used NHWC; we
# work in NCHW (1 input channel). "same"/stride-2 conv with kernel 3 needs
# padding=1. The encoder outputs z_mean and z_log_var.
import torch
from torch import nn

latent_dim = 2

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.dense = nn.Linear(7 * 7 * 64, 16)
        self.z_mean = nn.Linear(16, latent_dim)
        self.z_log_var = nn.Linear(16, latent_dim)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = torch.relu(self.dense(x))
        return self.z_mean(x), self.z_log_var(x)

encoder = Encoder()

In [ ]:
print(encoder)  # PyTorch: there's no Keras .summary(); print the module.

In [ ]:
# PyTorch: the Sampler is the reparameterization trick. We draw standard-normal
# noise with torch.randn and use exp(0.5 * z_log_var) as the std.
class Sampler(nn.Module):
    def forward(self, z_mean, z_log_var):
        epsilon = torch.randn_like(z_mean)
        return z_mean + torch.exp(0.5 * z_log_var) * epsilon

In [ ]:
# PyTorch: decoder as an nn.Module. Conv2DTranspose -> nn.ConvTranspose2d; for
# stride-2 "same" upsampling with kernel 3 we use padding=1, output_padding=1.
# We keep the final sigmoid here since the reconstruction loss below uses
# binary_crossentropy on probabilities.
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.Linear(latent_dim, 7 * 7 * 64)
        self.deconv1 = nn.ConvTranspose2d(64, 64, 3, stride=2, padding=1, output_padding=1)
        self.deconv2 = nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1)
        self.conv_out = nn.Conv2d(32, 1, 3, padding=1)

    def forward(self, x):
        x = torch.relu(self.dense(x))
        x = x.view(-1, 64, 7, 7)
        x = torch.relu(self.deconv1(x))
        x = torch.relu(self.deconv2(x))
        return torch.sigmoid(self.conv_out(x))

decoder = Decoder()

In [ ]:
print(decoder)  # PyTorch: print the module in place of Keras .summary().

In [ ]:
# PyTorch: the VAE wraps encoder + decoder + sampler. Keras's compute_loss
# (reconstruction + KL) is implemented explicitly below in the training loop;
# here forward() just returns everything the loss needs.
class VAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.sampler = Sampler()

    def forward(self, x):
        z_mean, z_log_var = self.encoder(x)
        reconstruction = self.decoder(self.sampler(z_mean, z_log_var))
        return reconstruction, z_mean, z_log_var

In [ ]:
# PyTorch: load MNIST via torchvision (no keras.datasets), concatenate train+test
# and scale to [0, 1]. Data is laid out as NCHW (channel dim = 1).
import numpy as np
from torchvision.datasets import MNIST

mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
mnist_digits = np.concatenate(
    [mnist_train.data.numpy(), mnist_test.data.numpy()], axis=0
)
mnist_digits = np.expand_dims(mnist_digits, 1).astype("float32") / 255

device = "cuda" if torch.cuda.is_available() else "cpu"
vae = VAE(encoder, decoder).to(device)
# PyTorch: compile() + fit() become an explicit training loop. The VAE loss is
# reconstruction (per-pixel BCE summed over H,W, averaged over the batch) plus the
# KL divergence, exactly as in Keras's compute_loss.
optimizer = torch.optim.Adam(vae.parameters())

x_train = torch.from_numpy(mnist_digits)
batch_size = 128
for epoch in range(30):
    permutation = torch.randperm(len(x_train))
    for i in range(0, len(x_train), batch_size):
        idx = permutation[i : i + batch_size]
        x = x_train[idx].to(device)
        reconstruction, z_mean, z_log_var = vae(x)
        reconstruction_loss = torch.mean(
            torch.sum(
                nn.functional.binary_cross_entropy(
                    reconstruction, x, reduction="none"
                ),
                dim=(1, 2, 3),
            )
        )
        kl_loss = -0.5 * (
            1 + z_log_var - torch.square(z_mean) - torch.exp(z_log_var)
        )
        total_loss = reconstruction_loss + torch.mean(kl_loss)
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
    print(
        f"Epoch {epoch}: loss {total_loss.item():.2f} "
        f"reconstruction {reconstruction_loss.item():.2f} "
        f"kl {torch.mean(kl_loss).item():.4f}"
    )

In [ ]:
import matplotlib.pyplot as plt

n = 30
digit_size = 28
figure = np.zeros((digit_size * n, digit_size * n))

grid_x = np.linspace(-1, 1, n)
grid_y = np.linspace(-1, 1, n)[::-1]

# PyTorch: decoder.predict() -> eval() + no_grad() forward pass.
vae.eval()
for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = torch.tensor([[xi, yi]], dtype=torch.float32, device=device)
        with torch.no_grad():
            x_decoded = vae.decoder(z_sample).cpu().numpy()
        digit = x_decoded[0].reshape(digit_size, digit_size)
        figure[
            i * digit_size : (i + 1) * digit_size,
            j * digit_size : (j + 1) * digit_size,
        ] = digit

plt.figure(figsize=(15, 15))
start_range = digit_size // 2
end_range = n * digit_size + start_range
pixel_range = np.arange(start_range, end_range, digit_size)
sample_range_x = np.round(grid_x, 1)
sample_range_y = np.round(grid_y, 1)
plt.xticks(pixel_range, sample_range_x)
plt.yticks(pixel_range, sample_range_y)
plt.xlabel("z[0]")
plt.ylabel("z[1]")
plt.axis("off")
plt.imshow(figure, cmap="Greys_r")

### Diffusion models

#### The Oxford Flowers dataset

In [ ]:
# PyTorch: download and extract the Oxford Flowers archive ourselves (no
# keras.utils.get_file). The images land in a "jpg" subfolder.
import os
import tarfile
import urllib.request

fpath = os.path.join("./data", "flower_photos")
os.makedirs(fpath, exist_ok=True)
tgz_path = os.path.join("./data", "102flowers.tgz")
if not os.path.exists(os.path.join(fpath, "jpg")):
    urllib.request.urlretrieve(
        "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz",
        tgz_path,
    )
    with tarfile.open(tgz_path) as tar:
        tar.extractall(fpath)

In [ ]:
# PyTorch: image_dataset_from_directory -> ImageFolder-style loading. The flowers
# live in a single "jpg" folder (no class subdirs), so we use a small Dataset that
# globs the files, center-crops to a square, and resizes to image_size. Images are
# returned as NCHW uint8-valued float tensors in [0, 255] to match the book's
# pipeline (normalization happens later inside the model).
import glob
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

batch_size = 32
image_size = 128
images_dir = os.path.join(fpath, "jpg")

class FlowerDataset(Dataset):
    def __init__(self, root, image_size):
        self.files = sorted(glob.glob(os.path.join(root, "*.jpg")))
        self.image_size = image_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        # crop_to_aspect_ratio=True -> center square crop, then resize.
        side = min(img.size)
        img = TF.center_crop(img, side)
        img = TF.resize(img, [self.image_size, self.image_size])
        return TF.pil_to_tensor(img).float()  # NCHW, values in [0, 255]

flower_dataset = FlowerDataset(images_dir, image_size)
# drop_last=True mirrors drop_remainder=True.
dataset = DataLoader(
    flower_dataset, batch_size=batch_size, shuffle=True, drop_last=True
)

In [ ]:
from matplotlib import pyplot as plt

for batch in dataset:
    # PyTorch: tensors are NCHW; permute to HWC for matplotlib.
    img = batch[0].permute(1, 2, 0).numpy()
    break
plt.imshow(img.astype("uint8"))

#### A U-Net denoising autoencoder

In [ ]:
# PyTorch: the U-Net becomes nn.Modules (NCHW). The residual block adds a 1x1
# projection only when the channel count changes; BatchNormalization with
# center=False, scale=False -> BatchNorm2d(affine=False). Skip connections are
# concatenated on the channel dim. The final conv is zero-initialized.
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_width, width):
        super().__init__()
        self.project = nn.Conv2d(in_width, width, 1) if in_width != width else None
        self.norm = nn.BatchNorm2d(in_width, affine=False)
        self.conv1 = nn.Conv2d(in_width, width, 3, padding=1)
        self.conv2 = nn.Conv2d(width, width, 3, padding=1)

    def forward(self, x):
        residual = self.project(x) if self.project is not None else x
        x = self.norm(x)
        x = F.silu(self.conv1(x))  # "swish" == silu
        x = self.conv2(x)
        return x + residual

class UNet(nn.Module):
    def __init__(self, image_size, widths, block_depth):
        super().__init__()
        self.image_size = image_size
        self.widths = widths
        self.block_depth = block_depth

        self.in_conv = nn.Conv2d(3, widths[0], 1)

        # Down path. After concatenating the upsampled noise rate (1 channel) to
        # the stem features, the first block sees widths[0] + 1 channels.
        self.down_blocks = nn.ModuleList()
        in_w = widths[0] + 1
        self.skip_widths = []
        for width in widths[:-1]:
            for _ in range(block_depth):
                self.down_blocks.append(ResidualBlock(in_w, width))
                self.skip_widths.append(width)
                in_w = width

        self.mid_blocks = nn.ModuleList()
        for _ in range(block_depth):
            self.mid_blocks.append(ResidualBlock(in_w, widths[-1]))
            in_w = widths[-1]

        # Up path: pop skips (LIFO) and concatenate before each block.
        self.up_blocks = nn.ModuleList()
        skips = list(self.skip_widths)
        for width in reversed(widths[:-1]):
            for _ in range(block_depth):
                skip_w = skips.pop()
                self.up_blocks.append(ResidualBlock(in_w + skip_w, width))
                in_w = width

        self.out_conv = nn.Conv2d(in_w, 3, 1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, noisy_images, noise_rates):
        x = self.in_conv(noisy_images)
        # UpSampling2D(image_size, nearest) broadcasts the (B,1,1,1) noise rate to
        # the full feature-map resolution, then we concatenate on the channel dim.
        n = noise_rates.expand(-1, -1, self.image_size, self.image_size)
        x = torch.cat([x, n], dim=1)

        skips = []
        bi = 0
        for width in self.widths[:-1]:
            for _ in range(self.block_depth):
                x = self.down_blocks[bi](x)
                bi += 1
                skips.append(x)
            x = F.avg_pool2d(x, 2)

        for block in self.mid_blocks:
            x = block(x)

        bi = 0
        for width in reversed(self.widths[:-1]):
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            for _ in range(self.block_depth):
                x = torch.cat([x, skips.pop()], dim=1)
                x = self.up_blocks[bi](x)
                bi += 1

        return self.out_conv(x)

def get_model(image_size, widths, block_depth):
    return UNet(image_size, widths, block_depth)

#### The concepts of diffusion time and diffusion schedule

In [ ]:
# PyTorch: keras.ops.* -> torch.*. diffusion_times is a tensor.
def diffusion_schedule(
    diffusion_times,
    min_signal_rate=0.02,
    max_signal_rate=0.95,
):
    start_angle = torch.acos(torch.tensor(max_signal_rate))
    end_angle = torch.acos(torch.tensor(min_signal_rate))
    diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)
    signal_rates = torch.cos(diffusion_angles)
    noise_rates = torch.sin(diffusion_angles)
    return noise_rates, signal_rates

In [ ]:
diffusion_times = torch.arange(0.0, 1.0, 0.01)
noise_rates, signal_rates = diffusion_schedule(diffusion_times)

# PyTorch: ops.convert_to_numpy -> .numpy().
diffusion_times = diffusion_times.numpy()
noise_rates = noise_rates.numpy()
signal_rates = signal_rates.numpy()

plt.plot(diffusion_times, noise_rates, label="Noise rate")
plt.plot(diffusion_times, signal_rates, label="Signal rate")

plt.xlabel("Diffusion time")
plt.legend()

#### The training process

In [ ]:
# PyTorch: DiffusionModel as an nn.Module. Keras's Normalization layer (adapted
# over the data) becomes registered buffers (mean/std) we fill in below. The MAE
# loss is computed explicitly in the training loop; here forward() returns the
# pieces it needs. generate() runs the reverse diffusion sampling loop.
class DiffusionModel(nn.Module):
    def __init__(self, image_size, widths, block_depth):
        super().__init__()
        self.image_size = image_size
        self.denoising_model = get_model(image_size, widths, block_depth)
        # Normalizer statistics over the channel dim (NCHW), filled by adapt().
        self.register_buffer("norm_mean", torch.zeros(1, 3, 1, 1))
        self.register_buffer("norm_std", torch.ones(1, 3, 1, 1))

    def adapt(self, dataloader):
        # PyTorch: replicates keras.layers.Normalization().adapt() — compute the
        # per-channel mean/variance across the whole dataset.
        total = 0
        s = torch.zeros(3)
        sq = torch.zeros(3)
        for batch in dataloader:
            b = batch.permute(0, 2, 3, 1).reshape(-1, 3)
            total += b.shape[0]
            s += b.sum(dim=0)
            sq += (b * b).sum(dim=0)
        mean = s / total
        var = sq / total - mean ** 2
        self.norm_mean.copy_(mean.view(1, 3, 1, 1))
        self.norm_std.copy_(var.clamp_min(1e-12).sqrt().view(1, 3, 1, 1))

    def normalize(self, images):
        return (images - self.norm_mean) / self.norm_std

    def denoise(self, noisy_images, noise_rates, signal_rates):
        pred_noise_masks = self.denoising_model(noisy_images, noise_rates)
        pred_images = (
            noisy_images - noise_rates * pred_noise_masks
        ) / signal_rates
        return pred_images, pred_noise_masks

    def forward(self, images):
        images = self.normalize(images)
        b = images.shape[0]
        noise_masks = torch.randn(
            (b, 3, self.image_size, self.image_size), device=images.device
        )
        diffusion_times = torch.rand((b, 1, 1, 1), device=images.device)
        noise_rates, signal_rates = diffusion_schedule(diffusion_times)
        noisy_images = signal_rates * images + noise_rates * noise_masks
        pred_images, pred_noise_masks = self.denoise(
            noisy_images, noise_rates, signal_rates
        )
        return pred_images, pred_noise_masks, noise_masks

    @torch.no_grad()
    def generate(self, num_images, diffusion_steps):
        device = self.norm_mean.device
        noisy_images = torch.randn(
            (num_images, 3, self.image_size, self.image_size), device=device
        )
        step_size = 1.0 / diffusion_steps
        for step in range(diffusion_steps):
            diffusion_times = (
                torch.ones((num_images, 1, 1, 1), device=device) - step * step_size
            )
            noise_rates, signal_rates = diffusion_schedule(diffusion_times)
            pred_images, pred_noises = self.denoise(
                noisy_images, noise_rates, signal_rates
            )
            next_diffusion_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = diffusion_schedule(
                next_diffusion_times
            )
            noisy_images = (
                next_signal_rates * pred_images + next_noise_rates * pred_noises
            )
        images = self.norm_mean + pred_images * self.norm_std
        return torch.clip(images, 0.0, 255.0)

#### The generation process

#### Visualizing results with a custom callback

In [ ]:
# PyTorch: the Keras callback becomes a plain function we call after each epoch.
def visualize(model, diffusion_steps=20, num_rows=3, num_cols=6):
    model.eval()
    generated_images = model.generate(
        num_images=num_rows * num_cols,
        diffusion_steps=diffusion_steps,
    )

    plt.figure(figsize=(num_cols * 2.0, num_rows * 2.0))
    for row in range(num_rows):
        for col in range(num_cols):
            i = row * num_cols + col
            plt.subplot(num_rows, num_cols, i + 1)
            # NCHW -> HWC for matplotlib.
            img = generated_images[i].permute(1, 2, 0).cpu().numpy().astype("uint8")
            plt.imshow(img)
            plt.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close()

#### It's go time!

In [ ]:
model = DiffusionModel(image_size, widths=[32, 64, 96, 128], block_depth=2).to(device)
# PyTorch: normalizer.adapt() over the dataset (computed on CPU tensors).
model.adapt(dataset)

In [ ]:
# PyTorch: compile() -> create the optimizer directly. AdamW with an inverse
# time-decay LR schedule (LambdaLR matching Keras's InverseTimeDecay formula).
# Keras's use_ema/ema_overwrite_frequency become an EMA of the parameters we
# maintain manually during training.
import copy

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def inverse_time_decay(step, initial=1e-3, decay_steps=1000, decay_rate=0.1):
    # LR factor relative to the initial LR (LambdaLR multiplies the base LR).
    return 1.0 / (1.0 + decay_rate * (step / decay_steps))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=inverse_time_decay)
ema_model = copy.deepcopy(model)
ema_decay = 0.999

In [ ]:
# PyTorch: model.fit() with callbacks -> explicit training loop. We use the MAE
# loss between predicted and true noise, maintain the EMA copy, run the
# visualization each epoch, and checkpoint the best (lowest-loss) weights.
loss_fn = nn.L1Loss()
best_loss = float("inf")

for epoch in range(100):
    model.train()
    running, batches = 0.0, 0
    for batch in dataset:
        batch = batch.to(device)
        _, pred_noise_masks, noise_masks = model(batch)
        loss = loss_fn(pred_noise_masks, noise_masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        # Maintain the EMA of the weights (Keras's use_ema).
        with torch.no_grad():
            for ema_p, p in zip(ema_model.parameters(), model.parameters()):
                ema_p.mul_(ema_decay).add_(p, alpha=1 - ema_decay)
        running += loss.item()
        batches += 1
    epoch_loss = running / max(batches, 1)
    print(f"Epoch {epoch}: loss {epoch_loss:.4f}")
    visualize(model)
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), "diffusion_model.weights.pt")

### Text-to-image models

In [ ]:
# PyTorch: the rest of this chapter does inference only. Disabling autograd keeps
# memory down. (This was already PyTorch-specific in the book.)
import torch

torch.set_grad_enabled(False)

In [ ]:
# PyTorch: keras_hub's TextToImage Stable Diffusion preset -> HuggingFace
# diffusers. We load Stable Diffusion 3 Medium via StableDiffusion3Pipeline; the
# pipeline returns PIL images. Note: this swaps keras_hub for diffusers but keeps
# the same model and prompt.
from diffusers import StableDiffusion3Pipeline

height, width = 512, 512
task = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    torch_dtype=torch.float16,
)
task = task.to("cuda" if torch.cuda.is_available() else "cpu")
prompt = "A NASA astraunaut riding an origami elephant in New York City"
task(prompt, height=height, width=width).images[0]

In [ ]:
# PyTorch: negative prompts are passed as a keyword argument in diffusers.
task(
    prompt,
    negative_prompt="blue color",
    height=height,
    width=width,
).images[0]

In [ ]:
import numpy as np
from PIL import Image

def display(images):
    return Image.fromarray(np.concatenate([np.array(im) for im in images], axis=1))

# PyTorch: num_steps -> num_inference_steps in diffusers.
display([
    task(prompt, height=height, width=width, num_inference_steps=x).images[0]
    for x in [5, 10, 15, 20, 25]
])

#### Exploring the latent space of a text-to-image model

In [ ]:
# PyTorch: explore the latent space by driving the diffusers pipeline's components
# directly. We encode the prompt with the pipeline's text encoders, then run the
# scheduler/transformer denoising loop and VAE decode by hand. This mirrors the
# book's encode_text / denoise_step / decode_step flow using diffusers internals.
import torch

def get_text_embeddings(prompt):
    device = task._execution_device
    (
        prompt_embeds,
        negative_prompt_embeds,
        pooled_prompt_embeds,
        negative_pooled_prompt_embeds,
    ) = task.encode_prompt(
        prompt=prompt,
        prompt_2=prompt,
        prompt_3=prompt,
        negative_prompt="",
        device=device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
    )
    # Return (positive, negative) pairs for both the sequence and pooled embeddings.
    return [
        prompt_embeds,
        negative_prompt_embeds,
        pooled_prompt_embeds,
        negative_pooled_prompt_embeds,
    ]

def denoise_with_text_embeddings(embeddings, num_steps=28, guidance_scale=7.0):
    device = task._execution_device
    prompt_embeds, negative_prompt_embeds, pooled, negative_pooled = embeddings
    task.scheduler.set_timesteps(num_steps, device=device)
    timesteps = task.scheduler.timesteps
    num_channels = task.transformer.config.in_channels
    latents = torch.randn(
        (1, num_channels, height // 8, width // 8),
        device=device,
        dtype=prompt_embeds.dtype,
    )
    combined_embeds = torch.cat([negative_prompt_embeds, prompt_embeds], dim=0)
    combined_pooled = torch.cat([negative_pooled, pooled], dim=0)
    for t in timesteps:
        latent_model_input = torch.cat([latents] * 2)
        timestep = t.expand(latent_model_input.shape[0])
        noise_pred = task.transformer(
            hidden_states=latent_model_input,
            timestep=timestep,
            encoder_hidden_states=combined_embeds,
            pooled_projections=combined_pooled,
            return_dict=False,
        )[0]
        noise_uncond, noise_text = noise_pred.chunk(2)
        noise_pred = noise_uncond + guidance_scale * (noise_text - noise_uncond)
        latents = task.scheduler.step(noise_pred, t, latents, return_dict=False)[0]
    latents = (latents / task.vae.config.scaling_factor) + task.vae.config.shift_factor
    image = task.vae.decode(latents, return_dict=False)[0]
    return image[0]

def scale_output(x):
    x = x.float().cpu().numpy()
    x = np.transpose(x, (1, 2, 0))  # CHW -> HWC
    x = np.clip((x + 1.0) / 2.0, 0.0, 1.0)
    return np.round(x * 255.0).astype("uint8")

embeddings = get_text_embeddings(prompt)
image = denoise_with_text_embeddings(embeddings)
scale_output(image)

In [ ]:
[x.shape for x in embeddings]

In [ ]:
# PyTorch: slerp/interpolation in torch instead of keras.ops. We interpolate the
# sequence embeddings (index 0 positive, index 2 pooled) and keep the negative
# embeddings fixed, matching the book's structure.
import torch

def slerp(t, v1, v2):
    v1, v2 = v1.float(), v2.float()
    v1_norm = torch.linalg.norm(v1.reshape(-1))
    v2_norm = torch.linalg.norm(v2.reshape(-1))
    dot = torch.sum(v1 * v2 / (v1_norm * v2_norm))
    theta_0 = torch.arccos(dot)
    sin_theta_0 = torch.sin(theta_0)
    theta_t = theta_0 * t
    sin_theta_t = torch.sin(theta_t)
    s0 = torch.sin(theta_0 - theta_t) / sin_theta_0
    s1 = sin_theta_t / sin_theta_0
    return s0 * v1 + s1 * v2

def interpolate_text_embeddings(e1, e2, start=0, stop=1, num=10):
    embeddings = []
    for t in np.linspace(start, stop, num):
        embeddings.append(
            [
                slerp(t, e1[0], e2[0]),
                e1[1],
                slerp(t, e1[2], e2[2]),
                e1[3],
            ]
        )
    return embeddings

In [ ]:
prompt1 = "A friendly dog looking up in a field of flowers"
prompt2 = "A horrifying, tentacled creature hovering over a field of flowers"
e1 = get_text_embeddings(prompt1)
e2 = get_text_embeddings(prompt2)

images = []
for et in interpolate_text_embeddings(e1, e2, start=0.5, stop=0.6, num=9):
    image = denoise_with_text_embeddings(et)
    images.append(Image.fromarray(scale_output(image)))
display(images)